# COINS Full Benchmark Notebook

Standalone benchmark notebook for COINS with Logistic Regression and SVM.

It implements same-family OOF scoring, CWMS, MSBS, and competitors: No Cleaning, Class-Proportional deletion, SMOTE, IW-SMOTE, and SW-approx.

COINS keeps every original label unchanged. It changes training influence through weights and adds MSBS minority rows.

## Full Run Setup

Default run: 15 cached datasets, LR/SVM, 10 seeds, 3 hidden-minority protocols, 6 methods.

For a quick execution check, launch with environment variable `COINS_SMOKE=1`.

In [1]:
from __future__ import annotations

import os
import time
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from scipy.stats import norm, wilcoxon
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
SMOKE = os.environ.get("COINS_SMOKE", "0") == "1"
print("root", ROOT)
print("smoke", SMOKE)


root /home/than-minh/project/DSP
smoke False


In [2]:
DATA_FILES = {
    "pima": "pima.parquet", "credit-g": "credit-g.parquet", "yeast": "yeast.parquet",
    "ecoli": "ecoli.parquet", "phoneme": "phoneme.parquet", "breast_cancer": "breast_cancer.parquet",
    "ilpd": "ilpd.parquet", "blood": "blood.parquet", "haberman": "haberman.parquet",
    "ionosphere": "ionosphere.parquet", "vehicle_bus": "vehicle_bus.parquet", "glass_float": "glass_float.parquet",
    "abalone": "abalone.parquet", "spambase": "spambase.parquet", "kc1": "kc1.parquet",
}

MINORITY_TEXT = {
    "pima": "tested_positive", "credit-g": "bad", "yeast": "MIT", "ecoli": "im", "phoneme": "nasal",
    "breast_cancer": "malignant", "ilpd": "no_disease", "blood": "donated", "haberman": "died",
    "ionosphere": "bad", "vehicle_bus": "bus", "glass_float": "window_float", "abalone": "rings_gt_10",
    "spambase": "spam", "kc1": "defective",
}

SEEDS_FULL = [13, 17, 23, 29, 31, 37, 41, 43, 47, 53]
PROTOCOLS_FULL = {
    "hidden_minority_low": (0.10, 0.05),
    "hidden_minority_medium": (0.30, 0.10),
    "hidden_minority_high": (0.40, 0.20),
}
MODELS = ["lr", "svm"]
METHODS = ["no_cleaning", "class_proportional", "smote", "iw_smote", "sw_approx", "coins"]
GOAL_MINORITY_RATIO = 0.15
BUDGET_RATIO = 0.10

DATA_NAMES = ["pima"] if SMOKE else list(DATA_FILES)
SEEDS = [13] if SMOKE else SEEDS_FULL
PROTOCOLS = {"hidden_minority_medium": PROTOCOLS_FULL["hidden_minority_medium"]} if SMOKE else PROTOCOLS_FULL

print(DATA_NAMES)
print(SEEDS)
print(PROTOCOLS)

['pima', 'credit-g', 'yeast', 'ecoli', 'phoneme', 'breast_cancer', 'ilpd', 'blood', 'haberman', 'ionosphere', 'vehicle_bus', 'glass_float', 'abalone', 'spambase', 'kc1']
[13, 17, 23, 29, 31, 37, 41, 43, 47, 53]
{'hidden_minority_low': (0.1, 0.05), 'hidden_minority_medium': (0.3, 0.1), 'hidden_minority_high': (0.4, 0.2)}


## Core Data And Model Helpers

In [3]:
def read_data(name: str) -> tuple[pd.DataFrame, np.ndarray]:
    """Load one cached parquet dataset and convert its minority class to label 1.

    The cached files keep the class column as the final column. The notebook uses
    `MINORITY_TEXT` so every dataset follows the same binary convention:
    minority = 1 and majority = 0.
    """
    df = pd.read_parquet(DATA_DIR / DATA_FILES[name])
    label_col = df.columns[-1]
    y = (df[label_col] == MINORITY_TEXT[name]).astype(int).to_numpy()
    X = df.drop(columns=[label_col])
    return X, y


def induce_ratio(
    X: pd.DataFrame,
    y: np.ndarray,
    goal_ratio: float,
    rng: np.random.Generator,
) -> tuple[pd.DataFrame, np.ndarray]:
    """Subsample the minority class to create the requested imbalance ratio.

    The benchmark follows the paper setup: reduce the minority side before noise
    injection so hidden-minority flips can erode an already small class.
    """
    min_idx = np.where(y == 1)[0]
    maj_idx = np.where(y == 0)[0]
    if len(min_idx) / len(y) <= goal_ratio:
        return X.reset_index(drop=True), y.copy()
    keep_min_n = min(len(min_idx), max(2, int((goal_ratio / (1 - goal_ratio)) * len(maj_idx))))
    keep_min = rng.choice(min_idx, size=keep_min_n, replace=False)
    keep = np.concatenate([maj_idx, keep_min])
    rng.shuffle(keep)
    return X.iloc[keep].reset_index(drop=True), y[keep].copy()


def add_noise(
    y_clean: np.ndarray,
    min_to_maj: float,
    maj_to_min: float,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray]:
    """Inject asymmetric training-label noise and return noisy labels plus mask.

    `min_to_maj` is the hidden-minority corruption rate. It should be larger
    than `maj_to_min` in the COINS operating condition.
    """
    y = y_clean.copy()
    mask = np.zeros(len(y), dtype=bool)
    for i, val in enumerate(y_clean):
        p = min_to_maj if val == 1 else maj_to_min
        if rng.random() < p:
            y[i] = 1 - y[i]
            mask[i] = True
    return y, mask


def preprocessor_for(X: pd.DataFrame) -> ColumnTransformer:
    """Build a tabular preprocessor for numeric and categorical columns.

    Numeric columns use median imputation and standard scaling. Categorical
    columns use most-frequent imputation and one-hot encoding. The transformed
    array is shared by all benchmark methods for a fair comparison.
    """
    cat = list(X.select_dtypes(include=["object", "category", "bool"]).columns)
    num = [c for c in X.columns if c not in cat]
    try:
        enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        enc = OneHotEncoder(handle_unknown="ignore", sparse=False)
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", enc)]), cat),
    ])


def split_data(
    name: str,
    seed: int,
    min_to_maj: float,
    maj_to_min: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Create one benchmark train/test split with imbalance and noisy train labels.

    The test set remains clean. Only the training labels are corrupted, matching
    the paper's controlled hidden-minority noise setup.
    """
    rng = np.random.default_rng(seed)
    X_raw, y_raw = read_data(name)
    X_tr_df, X_te_df, y_tr_clean, y_te = train_test_split(
        X_raw, y_raw, test_size=0.25, stratify=y_raw, random_state=seed
    )
    X_tr_df, y_tr_clean = induce_ratio(X_tr_df.reset_index(drop=True), y_tr_clean, GOAL_MINORITY_RATIO, rng)
    X_te_df = X_te_df.reset_index(drop=True)
    prep = preprocessor_for(X_tr_df)
    X_tr = prep.fit_transform(X_tr_df).astype(float)
    X_te = prep.transform(X_te_df).astype(float)
    y_noisy, noisy_mask = add_noise(y_tr_clean, min_to_maj, maj_to_min, rng)
    return X_tr, X_te, y_tr_clean, y_noisy, y_te, noisy_mask


def make_model(name: str, seed: int, balanced: bool = False) -> LogisticRegression | SVC:
    """Create the LR or SVM model used by the benchmark.

    `balanced=True` is used only for same-family OOF scoring. Final models are
    trained with explicit method-specific data/weights.
    """
    if name == "lr":
        return LogisticRegression(class_weight="balanced" if balanced else None, max_iter=1000, random_state=seed)
    if name == "svm":
        return SVC(kernel="rbf", probability=True, class_weight="balanced" if balanced else None, random_state=seed)
    raise ValueError(name)


In [4]:
def min_scores(model: LogisticRegression | SVC, X: np.ndarray) -> np.ndarray:
    """Return model probability for the minority class label 1."""
    return model.predict_proba(X)[:, list(model.classes_).index(1)]


def metrics_for(model: LogisticRegression | SVC, X_te: np.ndarray, y_te: np.ndarray) -> dict[str, float]:
    """Compute presentation metrics on the clean test set."""
    pred = model.predict(X_te)
    score = min_scores(model, X_te)
    return {
        "balanced_accuracy": balanced_accuracy_score(y_te, pred),
        "accuracy": accuracy_score(y_te, pred),
        "macro_f1": f1_score(y_te, pred, average="macro", zero_division=0),
        "minority_precision": precision_score(y_te, pred, pos_label=1, zero_division=0),
        "minority_recall": recall_score(y_te, pred, pos_label=1, zero_division=0),
        "majority_recall": recall_score(y_te, pred, pos_label=0, zero_division=0),
        "pr_auc": average_precision_score(y_te, score),
    }


def fit_eval(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_te: np.ndarray,
    y_te: np.ndarray,
    model_name: str,
    seed: int,
    weight: np.ndarray | None = None,
) -> dict[str, float]:
    """Fit one final LR/SVM model and evaluate it on clean test data."""
    model = make_model(model_name, seed)
    model.fit(X_tr, y_tr, sample_weight=weight) if weight is not None else model.fit(X_tr, y_tr)
    return metrics_for(model, X_te, y_te)


def oof_loss(X: np.ndarray, y: np.ndarray, model_name: str, seed: int) -> np.ndarray:
    """Compute out-of-fold negative log probability of each observed label.

    Class-proportional deletion uses this as a generic suspiciousness score.
    It is intentionally not COINS-specific.
    """
    out = np.zeros(len(y))
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for tr, va in cv.split(X, y):
        m = make_model(model_name, seed)
        m.fit(X[tr], y[tr])
        p = m.predict_proba(X[va])
        cols = [list(m.classes_).index(v) for v in y[va]]
        out[va] = -np.log(np.clip(p[np.arange(len(va)), cols], 1e-12, 1.0))
    return out


def same_family_oof(X: np.ndarray, y: np.ndarray, model_name: str, seed: int) -> np.ndarray:
    """Compute COINS same-family OOF minority scores for majority-labeled rows.

    LR final model receives scores from balanced LR. SVM final model receives
    scores from balanced SVM. Minority-labeled rows are set to NaN because COINS
    only scores the majority pool for hidden-minority candidates.
    """
    out = np.full(len(y), np.nan)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    for tr, va in cv.split(X, y):
        m = make_model(model_name, seed, balanced=True)
        m.fit(X[tr], y[tr])
        out[va] = min_scores(m, X[va])
    out[y == 1] = np.nan
    return out


## COINS And Competitor Implementations

In [5]:
def class_prop(
    X: np.ndarray,
    y: np.ndarray,
    loss: np.ndarray,
    budget: int,
) -> tuple[np.ndarray, np.ndarray, int]:
    """Delete high-loss rows with budget split proportionally by observed class."""
    selected: list[int] = []
    for lab in np.unique(y):
        idx = np.where(y == lab)[0]
        n = max(1, int(round(budget * len(idx) / len(y))))
        selected.extend(idx[np.argsort(loss[idx])[::-1]][:n].tolist())
    selected = np.array(selected[:budget], dtype=int)
    keep = np.ones(len(y), dtype=bool)
    keep[selected] = False
    return X[keep], y[keep], len(selected)


def msbs(
    X: np.ndarray,
    y: np.ndarray,
    scores: np.ndarray,
    budget: int,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, int]:
    """Generate MSBS synthetic minority rows near suspicious majority seeds.

    Anchors come from observed minority rows. Seeds come from majority-labeled
    rows with probability proportional to their OOF minority score.
    """
    rng = np.random.default_rng(seed)
    min_idx = np.where(y == 1)[0]
    maj_idx = np.where(y == 0)[0]
    if len(min_idx) == 0 or len(maj_idx) == 0:
        return X.copy(), y.copy(), 0
    vals = np.nan_to_num(scores[maj_idx], nan=0.0)
    probs = np.full(len(maj_idx), 1 / len(maj_idx)) if vals.sum() <= 0 else vals / vals.sum()
    synth: list[np.ndarray] = []
    for _ in range(budget):
        a = X[rng.choice(min_idx)]
        b = X[rng.choice(maj_idx, p=probs)]
        lam = rng.random()
        synth.append(a + lam * (b - a))
    return np.vstack([X, np.asarray(synth)]), np.concatenate([y, np.ones(len(synth), dtype=int)]), len(synth)


def coins_run(
    X: np.ndarray,
    y: np.ndarray,
    X_te: np.ndarray,
    y_te: np.ndarray,
    model_name: str,
    seed: int,
    budget: int,
) -> dict[str, float]:
    """Run full COINS: same-family OOF scoring, CWMS weights, and MSBS rows."""
    scores = same_family_oof(X, y, model_name, seed)
    X_aug, y_aug, n = msbs(X, y, scores, budget, seed)
    w = np.ones(len(y))
    maj = y == 0
    w[maj] = 1 - np.nan_to_num(scores[maj], nan=0.0)
    weight = np.concatenate([np.clip(w, 0, 1), np.ones(n)])
    ans = fit_eval(X_aug, y_aug, X_te, y_te, model_name, seed, weight)
    ans.update({"n_synthetic": n, "deleted": 0, "mean_cwms_weight": float(w.mean())})
    return ans


def smote_run(
    X: np.ndarray,
    y: np.ndarray,
    X_te: np.ndarray,
    y_te: np.ndarray,
    model_name: str,
    seed: int,
    budget: int,
) -> dict[str, float]:
    """Run standard SMOTE with the same synthetic-sample budget as COINS."""
    from imblearn.over_sampling import SMOTE
    n_min = int((y == 1).sum())
    n_maj = int((y == 0).sum())
    goal = min(n_min + budget, n_maj)
    if n_min < 2 or goal <= n_min:
        ans = fit_eval(X, y, X_te, y_te, model_name, seed)
        ans.update({"n_synthetic": 0, "deleted": 0})
        return ans
    sampler = SMOTE(sampling_strategy={1: goal}, k_neighbors=min(5, n_min - 1), random_state=seed)
    X_aug, y_aug = sampler.fit_resample(X, y)
    ans = fit_eval(X_aug, y_aug, X_te, y_te, model_name, seed)
    ans.update({"n_synthetic": len(y_aug) - len(y), "deleted": 0})
    return ans


In [6]:
def iw_smote_gen(
    X: np.ndarray,
    y: np.ndarray,
    budget: int,
    seed: int,
    lamda: int = 30,
) -> tuple[np.ndarray, np.ndarray, int]:
    """Generate IW-SMOTE samples using an under-bagged tree noise filter.

    This is a compact notebook implementation of the repository baseline. It
    filters high-error rows, then allocates synthesis to retained minority rows
    according to their ensemble error rate.
    """
    rng = np.random.default_rng(seed)
    min_idx = np.where(y == 1)[0]
    maj_idx = np.where(y == 0)[0]
    if len(min_idx) < 2 or len(maj_idx) < 1:
        return X.copy(), y.copy(), 0
    Xm, XM = X[min_idx], X[maj_idx]
    n_trees = max(1, int((len(XM) / len(Xm)) * lamda))
    pm = np.empty((len(Xm), n_trees), dtype=int)
    pM = np.empty((len(XM), n_trees), dtype=int)
    for i in range(n_trees):
        n = max(1, len(Xm) // 2)
        im = rng.choice(len(Xm), n, replace=False)
        iM = rng.choice(len(XM), n, replace=True)
        tree = DecisionTreeClassifier(random_state=int(rng.integers(0, 10000)))
        tree.fit(np.vstack([Xm[im], XM[iM]]), np.concatenate([np.ones(n), np.zeros(n)]))
        pm[:, i] = tree.predict(Xm)
        pM[:, i] = tree.predict(XM)
    em = (pm != 1).mean(axis=1)
    eM = (pM != 0).mean(axis=1)
    Xm2, XM2, em2 = Xm[em < 0.5], XM[eM < 0.5], em[em < 0.5]
    if len(Xm2) < 2:
        return np.vstack([XM2, Xm2]), np.concatenate([np.zeros(len(XM2)), np.ones(len(Xm2))]).astype(int), 0
    n_new = min(budget, max(0, len(XM2) - len(Xm2)))
    if n_new <= 0:
        return np.vstack([XM2, Xm2]), np.concatenate([np.zeros(len(XM2)), np.ones(len(Xm2))]).astype(int), 0
    weights = np.maximum(em2, 1 / n_trees)
    weights = weights / weights.sum()
    counts = (weights * n_new).astype(int)
    counts[np.argsort(weights)[::-1][: n_new - counts.sum()]] += 1
    synth: list[np.ndarray] = []
    for i, c in enumerate(counts):
        if c <= 0:
            continue
        d = np.linalg.norm(Xm2 - Xm2[i], axis=1)
        d[i] = np.inf
        nn = np.argsort(d)[: min(5, len(Xm2) - 1)]
        for _ in range(c):
            nb = Xm2[rng.choice(nn)]
            alpha = rng.random()
            synth.append(Xm2[i] + alpha * (nb - Xm2[i]))
    X_aug = np.vstack([XM2, Xm2, np.asarray(synth)]) if synth else np.vstack([XM2, Xm2])
    y_aug = np.concatenate([np.zeros(len(XM2)), np.ones(len(Xm2) + len(synth))]).astype(int)
    return X_aug, y_aug, len(synth)


def sw_gen(X: np.ndarray, y: np.ndarray, budget: int, seed: int) -> tuple[np.ndarray, np.ndarray, int]:
    """Generate SW-approx samples using RF leaf proximity as chaos proxy.

    The original SW Framework has no public code in this project. This notebook
    uses the repository's approximation: cleaner minority rows in RF-proximity
    neighborhoods receive more synthetic samples.
    """
    rng = np.random.default_rng(seed)
    min_idx = np.where(y == 1)[0]
    if len(min_idx) < 2:
        return X.copy(), y.copy(), 0
    rf = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=1)
    rf.fit(X, y)
    leaves = rf.apply(X)
    nn = NearestNeighbors(n_neighbors=min(6, len(X))).fit(leaves)
    _, near = nn.kneighbors(leaves[min_idx])
    chaos = np.array([np.mean(y[row[1:]] != 1) for row in near])
    weights = np.maximum(1 - chaos, 0.05)
    weights = weights / weights.sum()
    n_min, n_maj = len(min_idx), int((y == 0).sum())
    n_new = max(0, min(n_min + budget, n_maj) - n_min)
    if n_new <= 0:
        return X.copy(), y.copy(), 0
    counts = (weights * n_new).astype(int)
    counts[np.argsort(weights)[::-1][: n_new - counts.sum()]] += 1
    Xm = X[min_idx]
    synth: list[np.ndarray] = []
    for i, c in enumerate(counts):
        if c <= 0:
            continue
        d = np.linalg.norm(Xm - Xm[i], axis=1)
        d[i] = np.inf
        near2 = np.argsort(d)[: min(5, n_min - 1)]
        for _ in range(c):
            nb = Xm[rng.choice(near2)]
            alpha = rng.random()
            synth.append(Xm[i] + alpha * (nb - Xm[i]))
    return np.vstack([X, np.asarray(synth)]), np.concatenate([y, np.ones(len(synth), dtype=int)]), len(synth)


In [7]:
def run_method(
    method: str,
    X: np.ndarray,
    y: np.ndarray,
    X_te: np.ndarray,
    y_te: np.ndarray,
    model_name: str,
    seed: int,
    budget: int,
) -> dict[str, Any]:
    """Dispatch one benchmark method for one train/test split."""
    if method == "no_cleaning":
        ans = fit_eval(X, y, X_te, y_te, model_name, seed)
        ans.update({"n_synthetic": 0, "deleted": 0})
        return ans
    if method == "class_proportional":
        X2, y2, deleted = class_prop(X, y, oof_loss(X, y, model_name, seed), budget)
        ans = fit_eval(X2, y2, X_te, y_te, model_name, seed)
        ans.update({"n_synthetic": 0, "deleted": deleted})
        return ans
    if method == "smote":
        return smote_run(X, y, X_te, y_te, model_name, seed, budget)
    if method == "iw_smote":
        X2, y2, n = iw_smote_gen(X, y, budget, seed)
        ans = fit_eval(X2, y2, X_te, y_te, model_name, seed)
        ans.update({"n_synthetic": n, "deleted": max(0, len(y) - (len(y2) - n))})
        return ans
    if method == "sw_approx":
        X2, y2, n = sw_gen(X, y, budget, seed)
        ans = fit_eval(X2, y2, X_te, y_te, model_name, seed)
        ans.update({"n_synthetic": n, "deleted": 0})
        return ans
    if method == "coins":
        return coins_run(X, y, X_te, y_te, model_name, seed, budget)
    raise ValueError(method)


def run_combo(
    data_name: str,
    model_name: str,
    seed: int,
    proto_name: str,
    mn: float,
    mj: float,
) -> list[dict[str, Any]]:
    """Run every method for one dataset/model/seed/noise-protocol combination."""
    X, X_te, y_clean, y_noisy, y_te, mask = split_data(data_name, seed, mn, mj)
    budget = max(1, int(round(BUDGET_RATIO * len(y_noisy))))
    rows: list[dict[str, Any]] = []
    for method in METHODS:
        t0 = time.time()
        try:
            ans = run_method(method, X, y_noisy, X_te, y_te, model_name, seed, budget)
            err = ""
        except Exception as exc:
            ans = {k: np.nan for k in ["balanced_accuracy", "accuracy", "macro_f1", "minority_precision", "minority_recall", "majority_recall", "pr_auc"]}
            ans.update({"n_synthetic": 0, "deleted": 0})
            err = repr(exc)
        rows.append({
            "dataset": data_name,
            "model": model_name,
            "seed": seed,
            "noise_protocol": proto_name,
            "mn_to_maj": mn,
            "maj_to_min": mj,
            "method": method,
            "train_rows": len(y_noisy),
            "test_rows": len(y_te),
            "budget_count": budget,
            "true_noise_rate": float(mask.mean()),
            "runtime_sec": time.time() - t0,
            "error": err,
            **ans,
        })
    return rows


## Execute Benchmark

This cell stores results in memory and does not write CSV files.

In [8]:
rows = []
total = len(DATA_NAMES) * len(MODELS) * len(SEEDS) * len(PROTOCOLS)
i = 0
start = time.time()
for data_name in DATA_NAMES:
    for model_name in MODELS:
        for seed in SEEDS:
            for proto_name, (mn, mj) in PROTOCOLS.items():
                i += 1
                print(f"[{i}/{total}] {data_name} {model_name} seed={seed} {proto_name}", flush=True)
                rows.extend(run_combo(data_name, model_name, seed, proto_name, mn, mj))
results = pd.DataFrame(rows)
print("rows", len(results), "minutes", round((time.time() - start) / 60, 2))
results.head()

[1/900] pima lr seed=13 hidden_minority_low


[2/900] pima lr seed=13 hidden_minority_medium


[3/900] pima lr seed=13 hidden_minority_high


[4/900] pima lr seed=17 hidden_minority_low


[5/900] pima lr seed=17 hidden_minority_medium


...

[898/900] kc1 svm seed=53 hidden_minority_low


[899/900] kc1 svm seed=53 hidden_minority_medium


[900/900] kc1 svm seed=53 hidden_minority_high


rows 5400 minutes 20.13


,dataset,model,seed,noise_protocol,mn_to_maj,maj_to_min,method,train_rows,test_rows,budget_count,true_noise_rate,runtime_sec,error,balanced_accuracy,accuracy,macro_f1,minority_precision,minority_recall,majority_recall,pr_auc,n_synthetic,deleted,mean_cwms_weight
0,pima,lr,13,hidden_minority_low,0.1,0.05,no_cleaning,441,192,44,0.072562,0.007504,,0.540239,0.671875,0.488303,0.700000,0.104478,0.976,0.670561,0,0,NaN
1,pima,lr,13,hidden_minority_low,0.1,0.05,class_proportional,441,192,44,0.072562,0.014519,,0.678806,0.739583,0.688109,0.680851,0.477612,0.880,0.662807,0,44,NaN
2,pima,lr,13,hidden_minority_low,0.1,0.05,smote,441,192,44,0.072562,0.133971,,0.614328,0.718750,0.608163,0.782609,0.268657,0.960,0.676608,44,0,NaN
3,pima,lr,13,hidden_minority_low,0.1,0.05,iw_smote,441,192,44,0.072562,0.157943,,0.772060,0.770833,0.758131,0.641975,0.776119,0.768,0.666475,44,109,NaN
4,pima,lr,13,hidden_minority_low,0.1,0.05,sw_approx,441,192,44,0.072562,0.139081,,0.609254,0.703125,0.606882,0.666667,0.298507,0.920,0.663803,44,0,NaN


In [9]:
metric_cols = ["balanced_accuracy", "macro_f1", "minority_precision", "minority_recall", "majority_recall", "pr_auc"]
summary = results.groupby(["model", "method"])[metric_cols].mean().reset_index().sort_values(["model", "balanced_accuracy"], ascending=[True, False])
summary

,model,method,balanced_accuracy,macro_f1,minority_precision,minority_recall,majority_recall,pr_auc
1,lr,coins,0.732660,0.702255,0.541962,0.717149,0.748171,0.637050
2,lr,iw_smote,0.726252,0.709757,0.574047,0.649923,0.802581,0.636764
0,lr,class_proportional,0.699631,0.699711,0.648397,0.483763,0.915499,0.647831
5,lr,sw_approx,0.660270,0.657457,0.698869,0.363418,0.957122,0.645011
4,lr,smote,0.645588,0.638065,0.711988,0.326877,0.964300,0.638473
3,lr,no_cleaning,0.601185,0.575924,0.691486,0.214579,0.987791,0.642796
8,svm,iw_smote,0.745932,0.736875,0.624884,0.651792,0.840073,0.658148
7,svm,coins,0.687820,0.678374,0.665906,0.427069,0.948570,0.655120
6,svm,class_proportional,0.671351,0.658679,0.658783,0.370537,0.972165,0.642094
11,svm,sw_approx,0.654660,0.645256,0.709640,0.334754,0.974565,0.660648


In [10]:
def stouffer_report(
    df: pd.DataFrame,
    model_name: str,
    base: str = "class_proportional",
    challenger: str = "coins",
) -> tuple[pd.DataFrame, dict[str, Any]]:
    """Run per-dataset Wilcoxon tests and combine them with Stouffer's Z.

    This avoids treating all seed/protocol rows as independent. Each dataset gets
    its own paired Wilcoxon test; dataset-level Z scores are then combined.
    """
    rows2: list[dict[str, Any]] = []
    sub = df[df.model == model_name]
    for data_name in sorted(sub.dataset.unique()):
        a = sub[(sub.dataset == data_name) & (sub.method == challenger)].set_index(["seed", "noise_protocol"])
        b = sub[(sub.dataset == data_name) & (sub.method == base)].set_index(["seed", "noise_protocol"])
        j = a[["balanced_accuracy"]].join(b[["balanced_accuracy"]], lsuffix="_coins", rsuffix="_base", how="inner").dropna()
        if len(j) < 2:
            rows2.append({"dataset": data_name, "n": len(j), "delta_ba": np.nan, "p": np.nan, "z": np.nan})
            continue
        d = j.balanced_accuracy_coins - j.balanced_accuracy_base
        try:
            _, p = wilcoxon(d, alternative="greater", zero_method="wilcox")
        except ValueError:
            p = 1.0 if d.mean() <= 0 else 0.5
        rows2.append({
            "dataset": data_name,
            "n": len(j),
            "delta_ba": float(d.mean()),
            "p": float(p),
            "z": float(norm.isf(min(max(p, 1e-15), 1 - 1e-15))),
        })
    detail = pd.DataFrame(rows2)
    z = detail.z.dropna().to_numpy()
    stat = z.sum() / np.sqrt(len(z)) if len(z) else np.nan
    return detail, {
        "model": model_name,
        "baseline": base,
        "challenger": challenger,
        "stouffer_z": stat,
        "stouffer_p": norm.sf(stat) if np.isfinite(stat) else np.nan,
        "significant_datasets_p005": int((detail.p < 0.05).sum()),
    }


for m in MODELS:
    detail, stat = stouffer_report(results, m)
    print(stat)
    display(detail)


{'model': 'lr', 'baseline': 'class_proportional', 'challenger': 'coins', 'stouffer_z': np.float64(9.993580065629923), 'stouffer_p': np.float64(8.130039353834052e-24), 'significant_datasets_p005': 10}


,dataset,n,delta_ba,p,z
0,abalone,30,0.056060,9.313226e-10,6.009354
1,blood,30,0.066807,6.519258e-09,5.685559
2,breast_cancer,30,-0.018721,9.984214e-01,-2.951998
3,credit-g,30,0.019460,1.578087e-03,2.952103
4,ecoli,30,0.003495,2.389621e-01,0.709645
5,glass_float,30,0.104167,2.111383e-05,4.094941
6,haberman,30,0.031681,3.075127e-04,3.424900
7,ilpd,30,0.066781,9.313226e-09,5.624297
8,ionosphere,30,-0.009375,8.457118e-01,-1.018214
9,kc1,30,0.056632,8.666533e-07,4.782265


{'model': 'svm', 'baseline': 'class_proportional', 'challenger': 'coins', 'stouffer_z': np.float64(7.466224811602646), 'stouffer_p': np.float64(4.126415094119126e-14), 'significant_datasets_p005': 10}


,dataset,n,delta_ba,p,z
0,abalone,30,0.021746,2.527442e-04,3.477831
1,blood,30,0.009579,3.705801e-02,1.785897
2,breast_cancer,30,-0.023382,9.999592e-01,-3.939839
3,credit-g,30,0.007175,2.214634e-02,2.011310
4,ecoli,30,0.054710,1.694183e-03,2.930115
5,glass_float,30,0.041667,1.326140e-04,3.647089
6,haberman,30,0.015380,5.614884e-02,1.587950
7,ilpd,30,0.009013,7.360566e-02,1.449452
8,ionosphere,30,-0.047991,9.966174e-01,-2.708184
9,kc1,30,-0.006831,8.644287e-01,-1.100435


## Interpretation Notes

- Full-mode results are rerun from cached datasets and can be used for presentation tables.
- Smoke-mode results are execution checks only.
- COINS is boundary repair for hidden-minority asymmetric noise, not general label cleaning.
- Original labels are never modified.